# Predicting Airline Satisfaction: E01

Kaggle Playground Series S6E10. Binary probability of `satisfaction`,
scored by ROC AUC. This notebook compares three LightGBM fits on fold
definition F1. The control repeats the standing champion. The zero arm
adds indicators for survey scores equal to 0. The capacity arm keeps
the control features and uses 2,000 trees. CatBoost is not refit.

The comparison and the 0.0005 paired-gap rule were written in
`docs/3_implementation_plan.md` before this run.

CPU kernel. Internet disabled. GPU is not used.


## 1. Environment


In [ ]:
import json
import platform
import shutil
import time
from pathlib import Path

import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

print("python", platform.python_version())
print({
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
})


## 2. Config


In [ ]:
SEED = 42
NOTEBOOK_VERSION = "v3"
TARGET = "satisfaction"
ID_COLUMN = "id"
ARRIVAL = "Arrival Delay in Minutes"
N_SPLITS = 5
PROMOTION_MARGIN = 0.0005
TREE_BUDGET = 500
CAPACITY_TREES = 2000
LEARNING_RATE = 0.05

CATEGORICAL_FEATURES = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
]

# Survey columns whose observed minimum is 0. Baggage handling is
# excluded (minimum 1). Delay columns are excluded because 0 is a
# measured delay, not a survey code.
ZERO_SURVEY_COLUMNS = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Checkin service",
    "Cleanliness",
]


def find_data_dir() -> Path:
    """Locate competition files without assuming one mount path.

    Returns:
        Directory that contains train.csv.

    Raises:
        FileNotFoundError: If train.csv is not on a tried path.
    """
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e10"),
        Path("/kaggle/input/playground-series-s6e10"),
        Path("../data"),
        Path("data"),
    ]
    for cand in candidates:
        if (cand / "train.csv").exists():
            return cand
    mount = Path("/kaggle/input")
    if mount.exists():
        hits = sorted(mount.rglob("train.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("train.csv not found in any known location")


DATA_DIR = find_data_dir()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("predictions")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA_DIR", DATA_DIR.resolve())
print("OUT", OUT.resolve())
print(
    "F1 = StratifiedKFold(5, shuffle=True, random_state=42). "
    f"An arm replaces the control only if its OOF AUC is higher and "
    f"the mean paired fold gap is at least {PROMOTION_MARGIN}."
)


## 3. Data


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
y = train[TARGET].astype(int)
feature_columns = [c for c in train.columns if c not in (ID_COLUMN, TARGET)]
assert feature_columns == [c for c in test.columns if c != ID_COLUMN]
assert list(sample.columns) == [ID_COLUMN, TARGET]
missing_zero = [c for c in ZERO_SURVEY_COLUMNS if c not in feature_columns]
assert not missing_zero, missing_zero
print(train.shape, test.shape, "positive", int(y.sum()))
print("zero indicators", len(ZERO_SURVEY_COLUMNS))


## 4. Fold-safe recipe

The missingness indicator uses the raw mask. The arrival-delay median
is fit on the training rows of the current fold only. A zero indicator
is a property of the row.


In [ ]:
def prepare_fold(
    train_frame: pd.DataFrame,
    apply_frame: pd.DataFrame,
    fit_index: np.ndarray,
    zero_indicators: bool,
) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    """Apply one recipe using one fold's training rows.

    Args:
        train_frame: Full training frame, raw columns only.
        apply_frame: Validation rows or the test frame.
        fit_index: Positions in train_frame used to fit the median.
        zero_indicators: When True, append one indicator per survey
            column listed in ZERO_SURVEY_COLUMNS.

    Returns:
        Transformed fit rows only, transformed apply_frame, and the median.
    """
    fit_rows = train_frame.iloc[fit_index]
    median = float(fit_rows[ARRIVAL].median())

    def transform(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame[feature_columns].copy()
        out["arrival_delay_missing"] = out[ARRIVAL].isna().astype(np.int8)
        out[ARRIVAL] = out[ARRIVAL].fillna(median)
        if zero_indicators:
            for col in ZERO_SURVEY_COLUMNS:
                out[f"{col}__is_zero"] = (out[col] == 0).astype(np.int8)
        return out

    return transform(fit_rows), transform(apply_frame), median


## 5. Model


In [ ]:
def fit_lightgbm(
    x_fit: pd.DataFrame,
    y_fit: pd.Series,
    n_estimators: int,
) -> LGBMClassifier:
    """Fit one LightGBM arm.

    Args:
        x_fit: Training-fold features.
        y_fit: Training-fold labels.
        n_estimators: Tree count for this arm.

    Returns:
        A fitted classifier.
    """
    frame = x_fit.copy()
    for col in CATEGORICAL_FEATURES:
        frame[col] = frame[col].astype("category")
    model = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=LEARNING_RATE,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(frame, y_fit, categorical_feature=CATEGORICAL_FEATURES)
    return model


def predict(model: LGBMClassifier, frame: pd.DataFrame) -> np.ndarray:
    """Predict the positive-class probability.

    Args:
        model: A fitted LightGBM classifier.
        frame: Features in the training column order.

    Returns:
        A one-dimensional probability vector.
    """
    frame = frame.copy()
    for col in CATEGORICAL_FEATURES:
        frame[col] = frame[col].astype("category")
    return model.predict_proba(frame)[:, 1]


## 6. Cross-validation

`lightgbm_control` and `lightgbm_capacity` share one feature matrix.
`lightgbm_zero` uses the indicator matrix from the same fold.


In [ ]:
ARMS = (
    ("lightgbm_control", False, TREE_BUDGET),
    ("lightgbm_zero", True, TREE_BUDGET),
    ("lightgbm_capacity", False, CAPACITY_TREES),
)

folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = {name: np.zeros(len(train), dtype=np.float64) for name, _, _ in ARMS}
test_pred = {name: np.zeros(len(test), dtype=np.float64) for name, _, _ in ARMS}
fold_auc = {name: [] for name in oof}
timings = {name: [] for name in oof}

for fold_id, (fit_idx, val_idx) in enumerate(folds.split(train, y), start=1):
    prepared = {}
    medians = {}
    for use_zero in (False, True):
        x_fit, x_val, median = prepare_fold(
            train, train.iloc[val_idx], fit_idx, use_zero
        )
        _, x_test, _ = prepare_fold(train, test, fit_idx, use_zero)
        prepared[use_zero] = (x_fit, x_val, x_test)
        medians[use_zero] = median
    assert medians[False] == medians[True]
    y_fit = y.iloc[fit_idx]
    y_val = y.iloc[val_idx]
    print(f"fold {fold_id} median {medians[False]:.4f} val rows {len(val_idx)}")
    for name, use_zero, n_estimators in ARMS:
        x_fit, x_val, x_test = prepared[use_zero]
        started = time.perf_counter()
        model = fit_lightgbm(x_fit, y_fit, n_estimators)
        val_hat = predict(model, x_val)
        test_hat = predict(model, x_test)
        elapsed = time.perf_counter() - started
        oof[name][val_idx] = val_hat
        test_pred[name] += test_hat / N_SPLITS
        score = float(roc_auc_score(y_val, val_hat))
        fold_auc[name].append(score)
        timings[name].append(elapsed)
        print(f"  {name} trees {n_estimators} fold AUC {score:.6f} seconds {elapsed:.1f}")

for name in oof:
    np.save(OUT / f"oof_{name}.npy", oof[name])
    np.save(OUT / f"test_{name}.npy", test_pred[name])
    print(
        name,
        "OOF", f"{roc_auc_score(y, oof[name]):.6f}",
        "fold mean", f"{float(np.mean(fold_auc[name])):.6f}",
        "fold std", f"{float(np.std(fold_auc[name])):.6f}",
        "seconds", f"{sum(timings[name]):.1f}",
    )


## 7. Promotion gate

The control must later match the stored B01 LightGBM OOF bit for bit.
That check happens after the kernel output is downloaded. This cell
only compares the three arms in this run.


In [ ]:
control_auc = float(roc_auc_score(y, oof["lightgbm_control"]))
control_folds = np.asarray(fold_auc["lightgbm_control"])
passed = []
arm_auc = {}
arm_gap = {}
for name in ("lightgbm_zero", "lightgbm_capacity"):
    auc = float(roc_auc_score(y, oof[name]))
    gap = float((np.asarray(fold_auc[name]) - control_folds).mean())
    arm_auc[name] = auc
    arm_gap[name] = gap
    ok = (auc > control_auc) and (gap >= PROMOTION_MARGIN)
    print(
        f"{name} OOF {auc:.6f} mean paired gap {gap:.6f} "
        f"clears {ok}"
    )
    if ok:
        passed.append(name)

if not passed:
    champion = "lightgbm_control"
elif len(passed) == 1:
    champion = passed[0]
else:
    champion = max(passed, key=lambda name: arm_auc[name])
print("champion", champion)
print("NOTEBOOK_VERSION", NOTEBOOK_VERSION)

chosen = test_pred[champion]
assert np.isfinite(chosen).all()
assert ((chosen >= 0.0) & (chosen <= 1.0)).all()
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN].to_numpy(),
    TARGET: chosen,
})
submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)
summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "fold": "F1",
    "control_oof_auc": control_auc,
    "arm_oof_auc": arm_auc,
    "mean_paired_gap": arm_gap,
    "promotion_margin": PROMOTION_MARGIN,
    "passed": passed,
    "champion": champion,
    "fold_auc": fold_auc,
    "seconds": {name: sum(values) for name, values in timings.items()},
}
(OUT / "e01_summary.json").write_text(json.dumps(summary, indent=2))
print("wrote", submission_path.name, len(submission))


## 8. Self-export

Last cell. Copies the executed notebook, which holds every output above
and none of this cell's own output.


In [ ]:
source = Path("/kaggle/working/__notebook__.ipynb")
if source.exists():
    dest = Path("/kaggle/working/executed_notebook.ipynb")
    shutil.copy(source, dest)
    exported = json.loads(dest.read_text())
    with_outputs = sum(1 for cell in exported["cells"] if cell.get("outputs"))
    print(
        f"self-export: {dest.name} — {len(exported['cells'])} cells, "
        f"{with_outputs} with outputs, {dest.stat().st_size / 1024:.0f} KB"
    )
else:
    print("no __notebook__.ipynb — self-export skipped")
